In [1]:
"""
================================================================================
Lateral Vehicle Dynamics — Lane Change Case Study
HT-MPC (homothetic, proposed) vs RT-MPC (rigid tube)

Three schemes, identical in everything except the tube scalings:
  HT-MPC:  alpha_k = ||e_k||_{P^{-1}},  tightening alpha_{j|k} from the
           closed-form scale sequence (Prop. 3 of the paper)        [proposed]
  HT-MPC with optimised scalings: the same alpha_{j|k} are DECISION
           VARIABLES, tied by alpha_{j+1} >= beta(alpha_j); an SOCP   [baseline]
           S. V. Rakovic, B. Kouvaritakis, R. Findeisen, M. Cannon,
           "Homothetic tube model predictive control", Automatica,
           vol. 48, no. 8, pp. 1631-1638, 2012.
  RT-MPC:  alpha_{j|k} = 1 for all j  (full worst-case tightening)

Both online problems are standard QPs (quadratic cost, linear constraints,
polytopic terminal set), solved with Clarabel (OSQP as fallback).

System
------
Lateral bicycle model linearised at v_x0 = 60 km/h, discretised with ZOH.
  States:  x = [e_y (m), e_psi (rad), v_y (m/s), psi_dot (rad/s)]
           e_y is the lateral position measured from the road centre.
  Input:   u = delta_s (rad)   (front steering angle)
  Ts = 0.1 s

Uncertainty  (Assumption 1 of the paper)
-----------
Speed mismatch Delta_v_x, bounded by |Delta_v_x| <= sqrt(chi2_{1,p}) * sigma_vx.
First-order Taylor expansion of the ZOH model in v_x:
  Delta_A = Delta_v_x * D_A,  D_A = dA_d/dv_x
  Delta_B = Delta_v_x * D_B,  D_B = dB_d/dv_x   (nonzero under ZOH)
  => Gamma_A, Gamma_B rank one (plus a tiny regularisation)
Additive disturbance w_k sampled inside the ellipsoid Ell(Gamma_w), so that
Assumption 1 holds at every step.

Terminal ingredients
--------------------
V_f(z) = z' P_f z with A_K' P_f A_K - P_f = -(Q_c + K' R_c K)   (paper eq.)
Z_f    = maximal positively invariant polytope of z+ = A_K z under the
         rigid tightening (alpha = 1), built in coordinates shifted to the
         current setpoint (z_ref, v_ref).

Lane change scenario
--------------------
Right lane: e_y = -1.75 m, left lane: e_y = +1.75 m (lane width 3.5 m).
Step change of the setpoint at k = T_lc.  Between switches the setpoint is
constant, so the guarantees of the paper apply in the shifted coordinates;
at the switch the controller re-centres its cost and terminal set.

Changes w.r.t. the previous version
-----------------------------------
 (a) P_f from the Lyapunov equation of A_K (was: DARE with different weights,
     which violated the terminal decrease condition).
 (b) Polytopic terminal set (QP) instead of an ellipsoidal one (SOCP).
 (c) Delta_B included (first-order ZOH sensitivity).
 (d) w_k sampled inside Ell(Gamma_w); no cap on alpha_k (any assumption
     violation is counted and reported).
 (e) Figure: the bands are the projected cross-sections by which each
     scheme tightens the constraints of the current step (alpha_k*Omega for
     HT-MPC, Omega for RT-MPC).  Both schemes have the same realized error;
     the difference is the tightening, which frees the steering in HT-MPC.

Figure produced
---------------
fig_lane_change_comparison_alt.png
  Lateral position along the road, with the projected cross-sections used
  to tighten the current step (alpha_k*Omega for HT-MPC, Omega for RT-MPC).
The baseline scheme is not drawn: its closed loop coincides with the
proposed one (see the printed comparison).
================================================================================
"""

import numpy as np
import scipy.linalg   as la
import scipy.signal   as sig
import scipy.stats    as stats
import scipy.optimize as opt
import cvxpy          as cp
import matplotlib.pyplot as plt
from matplotlib.lines   import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from dataclasses import dataclass
from itertools   import product as iprod

# ── matplotlib style (IEEE single-column) ─────────────────────────────────────
FP      = 8
ONE_COL = 3.5

plt.rcParams.update({
    "mathtext.fontset": "cm",
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":         FP,    "axes.labelsize":   FP,
    "axes.titlesize":    FP,    "xtick.labelsize":  FP - 1,
    "ytick.labelsize":   FP - 1,"legend.fontsize":  FP - 1,
    "legend.framealpha": 0.90,  "legend.frameon":   True,
    "legend.edgecolor":  "0.80","lines.linewidth":  1.0,
    "axes.linewidth":    0.6,   "grid.linewidth":   0.4,
    "grid.alpha":        0.40,  "savefig.dpi":      300,
    "savefig.bbox":      "tight","pdf.fonttype":    42,
    "ps.fonttype":       42,
})

C = {
    "p":      "#007E5C",     # HT-MPC  — green
    "r":      "#FF4500",     # RT-MPC  — orange-red
    "ref":    "#1D1D1DCA",   # reference — near-black
    "constr": "#A80000",     # constraints / road edges — dark red
}
LABELS = {"p": "HT-MPC", "r": "RT-MPC"}


# ══════════════════════════════════════════════════════════════════════════════
#  VEHICLE MODEL
# ══════════════════════════════════════════════════════════════════════════════

_m  = 1500.0;  _Iz = 2500.0
_lf = 1.2;     _lr = 1.6
_Cf = 60000.0; _Cr = 70000.0


def _Ac(vx):
    m, Iz, lf, lr, Cf, Cr = _m, _Iz, _lf, _lr, _Cf, _Cr
    return np.array([
        [0.,  vx,  1.,  0.],
        [0.,  0.,  0.,  1.],
        [0.,  0.,  -2.*(Cf+Cr)/(m*vx),
                   (-2.*(Cf*lf-Cr*lr)/(m*vx) - vx)],
        [0.,  0.,  -2.*(Cf*lf-Cr*lr)/(Iz*vx),
                   -2.*(Cf*lf**2+Cr*lr**2)/(Iz*vx)],
    ])


def _Bc():
    m, Iz, lf, Cf = _m, _Iz, _lf, _Cf
    return np.array([[0.], [0.], [2.*Cf/m], [2.*Cf*lf/Iz]])


def discretize(vx, Ts):
    sd = sig.StateSpace(_Ac(vx), _Bc(), np.eye(4),
                        np.zeros((4, 1))).to_discrete(Ts)   # ZOH
    return np.array(sd.A), np.array(sd.B)


def d_dvx(vx0, Ts, h=1e-4):
    """First-order sensitivities D_A = dA_d/dv_x and D_B = dB_d/dv_x."""
    Ap, Bp = discretize(vx0 + h, Ts)
    Am, Bm = discretize(vx0 - h, Ts)
    return (Ap - Am)/(2.*h), (Bp - Bm)/(2.*h)


# ══════════════════════════════════════════════════════════════════════════════
#  UNCERTAINTY MODEL  (Assumption 1)
# ══════════════════════════════════════════════════════════════════════════════

class UncertaintyModel:
    """
    Shapes Gamma_A = chi2_DA*Sigma_DA, Gamma_B = chi2_DB*Sigma_DB,
    Gamma_w = chi2_w*Sigma_w.  vec(.) is row-major here, consistent with
    Delta_A x = kron(I_n, x') vec(Delta_A).
    """
    def __init__(self, Sigma_DA, Sigma_DB, Sigma_w,
                 p_A, p_B, p_W, n, m, df_DA=1, df_DB=1):
        self.Sigma_DA = np.atleast_2d(Sigma_DA)
        self.Sigma_DB = np.atleast_2d(Sigma_DB)
        self.Sigma_w  = np.atleast_2d(Sigma_w)
        self.n = n;  self.m = m
        self.chi2_DA = float(stats.chi2.ppf(p_A, df=df_DA))
        self.chi2_DB = float(stats.chi2.ppf(p_B, df=df_DB))
        self.chi2_w  = float(stats.chi2.ppf(p_W, df=n))
        self.Gamma_w = self.chi2_w*self.Sigma_w
        print(f"  OK  chi2_DA={self.chi2_DA:.3f}  chi2_DB={self.chi2_DB:.3f}"
              f"  chi2_w={self.chi2_w:.3f}")

    def QA_at_x(self, x):
        L = np.kron(np.eye(self.n), x.reshape(1, -1))
        Q = L @ self.Sigma_DA @ L.T
        return 0.5*(Q + Q.T)

    def QB_at_u(self, u):
        L = np.kron(np.eye(self.n), u.reshape(1, -1))
        Q = L @ self.Sigma_DB @ L.T
        return 0.5*(Q + Q.T)


# ══════════════════════════════════════════════════════════════════════════════
#  VERTEX BOUND (Lemma 2) + SDP (eq. Qmax)
# ══════════════════════════════════════════════════════════════════════════════

def _box_vertices(H, h):
    d = H.shape[1]
    assert H.shape[0] == 2*d
    assert np.allclose(H, np.vstack([np.eye(d), -np.eye(d)]))
    return np.array(list(iprod(*zip(-h[d:], h[:d]))))


def compute_Qmax(unc, Hx, h_x, Hu, h_u):
    n = unc.n
    def _sdp(mats):
        Qv = cp.Variable((n, n), symmetric=True)
        cp.Problem(cp.Minimize(cp.trace(Qv)),
                   [Qv >> M + 1e-9*np.eye(n) for M in mats]
                   ).solve(solver="SCS", verbose=False, eps=1e-8,
                           max_iters=15_000)
        if Qv.value is None:
            return mats[0] + 1e-6*np.eye(n)
        return 0.5*(Qv.value + Qv.value.T) + 1e-9*np.eye(n)
    QA_max = _sdp([unc.QA_at_x(v) for v in _box_vertices(Hx, h_x)])
    QB_max = _sdp([unc.QB_at_u(v) for v in _box_vertices(Hu, h_u)])
    print(f"  OK  tr(Q^A_max)={np.trace(QA_max):.3e}  "
          f"tr(Q^B_max)={np.trace(QB_max):.3e}")
    return QA_max, QB_max


# ══════════════════════════════════════════════════════════════════════════════
#  RPI ELLIPSOID (Prop. 2) + CONTRACTION COEFFICIENT (Def. 1)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class RPIData:
    P_star:     np.ndarray
    P_star_inv: np.ndarray
    c_S:        float
    tau_star:   float
    S_bar:      np.ndarray


def compute_rpi(A_K, unc, QA_max, QB_max):
    # optimal weights (Prop. 1) and S_bar (eq. Smax)
    G = [unc.chi2_DA*QA_max, unc.chi2_DB*QB_max, unc.Gamma_w]
    a = [float(np.trace(Gi)) for Gi in G]
    s = sum(np.sqrt(ai) for ai in a)
    mu = [np.sqrt(ai)/s for ai in a]
    S_bar = sum(Gi/mi for Gi, mi in zip(G, mu))
    S_bar = 0.5*(S_bar + S_bar.T)

    def _stein(tau):
        P = la.solve_discrete_lyapunov(A_K/np.sqrt(tau), S_bar/(1.-tau))
        return 0.5*(P + P.T)

    # golden-section search for tau* (tr(P_tau) is strictly convex)
    rho2 = float(np.max(np.abs(la.eigvals(A_K))))**2
    lo, hi = rho2 + 1e-8, 1. - 1e-8
    gr = (np.sqrt(5.) - 1.)/2.
    x1 = hi - gr*(hi - lo);  f1 = np.trace(_stein(x1))
    x2 = lo + gr*(hi - lo);  f2 = np.trace(_stein(x2))
    for _ in range(400):
        if hi - lo < 1e-12: break
        if f1 < f2:
            hi, x2, f2 = x2, x1, f1; x1 = hi - gr*(hi - lo); f1 = np.trace(_stein(x1))
        else:
            lo, x1, f1 = x1, x2, f2; x2 = lo + gr*(hi - lo); f2 = np.trace(_stein(x2))
    tau_star = 0.5*(lo + hi)
    P_star = _stein(tau_star)
    Pnh = np.real(la.inv(la.sqrtm(P_star)))
    c_S = float(np.max(np.real(la.eigvals(Pnh @ S_bar @ Pnh))))/(1. - tau_star)
    assert 0. < c_S < 1., f"c_S={c_S} outside (0,1) — check Assumption 2."
    print(f"  OK  tau*={tau_star:.4f}  c_S={c_S:.6f}  "
          f"sqrt(P*[0,0])={np.sqrt(P_star[0, 0]):.4f} m")
    return RPIData(P_star=P_star, P_star_inv=la.inv(P_star),
                   c_S=c_S, tau_star=tau_star, S_bar=S_bar)


def alpha_seq(alpha_k, c_S, N):
    """Closed-form scale sequence (Prop. 3)."""
    j = np.arange(N + 1, dtype=float)
    return np.sqrt(np.maximum(1. - (1. - c_S)**j*(1. - alpha_k**2), 0.))


# ══════════════════════════════════════════════════════════════════════════════
#  TIGHTENING (eq. support) + POLYTOPIC TERMINAL SET
# ══════════════════════════════════════════════════════════════════════════════

def support_values(K, P_star, Hx, Hu):
    s_x = np.array([np.sqrt(float(h @ P_star @ h)) for h in Hx])
    KPK = K @ P_star @ K.T
    s_u = np.array([np.sqrt(float(h @ KPK @ h)) for h in Hu])
    return s_x, s_u


def steady_state(A, B, C_out, r_ref):
    n_, m_ = A.shape[0], B.shape[1];  p_ = C_out.shape[0]
    r = np.atleast_1d(float(r_ref))
    M   = np.block([[A - np.eye(n_), B], [C_out, np.zeros((p_, m_))]])
    rhs = np.concatenate([np.zeros(n_), r])
    sol, *_ = np.linalg.lstsq(M, rhs, rcond=None)
    return sol[:n_], sol[n_:]


class TerminalSet:
    """
    Maximal positively invariant polytope of zeta+ = A_K zeta, with
    zeta = z - z_ref, under the rigid tightening (alpha = 1):
        Hx (z_ref + zeta)    <= h_x - s_x
        Hu (v_ref + K zeta)  <= h_u - s_u
    Rows G A_K^t, t = 0..t*, are fixed; the right-hand side depends on the
    setpoint.  t* is chosen so that the set is finitely determined for every
    setpoint passed in `setpoints` (Gilbert & Tan, 1991).
    """
    def __init__(self, A_K, K, Hx, h_x, Hu, h_u, s_x, s_u, setpoints,
                 t_max=200):
        self.G0 = np.vstack([Hx, Hu @ K])
        self.Hx, self.Hu = Hx, Hu
        self.bx = h_x - s_x;  self.bu = h_u - s_u
        t_star = 0
        for (zr, vr) in setpoints:
            g0 = self.rhs0(zr, vr)
            assert np.all(g0 > 0), "setpoint not in the interior of Z(1)"
            t_star = max(t_star, self._determinedness(A_K, g0, t_max))
        self.t_star = t_star
        self.G = np.vstack([self.G0 @ np.linalg.matrix_power(A_K, t)
                            for t in range(t_star + 1)])
        print(f"  OK  terminal set: t*={t_star}, {self.G.shape[0]} rows")

    def rhs0(self, z_ref, v_ref):
        return np.concatenate([self.bx - self.Hx @ z_ref,
                               self.bu - self.Hu @ v_ref])

    def rhs(self, z_ref, v_ref):
        return np.tile(self.rhs0(z_ref, v_ref), self.t_star + 1)

    def _determinedness(self, A_K, g0, t_max):
        Gs, gs = [self.G0], [g0]
        for t in range(t_max):
            Gt = np.vstack(Gs);  gt = np.concatenate(gs)
            Gn = self.G0 @ np.linalg.matrix_power(A_K, t + 1)
            redundant = True
            for i, row in enumerate(Gn):
                res = opt.linprog(-row, A_ub=Gt, b_ub=gt,
                                  bounds=[(None, None)]*A_K.shape[0],
                                  method="highs")
                if (not res.success) or (-res.fun > g0[i] + 1e-9):
                    redundant = False; break
            if redundant:
                return t
            Gs.append(Gn);  gs.append(g0)
        raise RuntimeError("terminal set not finitely determined")


# ══════════════════════════════════════════════════════════════════════════════
#  OCP  (Definition 2) — same QP for both schemes, different tightening
# ══════════════════════════════════════════════════════════════════════════════

class TubeOCP:
    def __init__(self, A, B, Hx, h_x, Hu, h_u, s_x, s_u, term, N,
                 Q_c, R_c, P_f, name):
        n = A.shape[0];  m = B.shape[1]
        self.N, self.h_x, self.h_u, self.s_x, self.s_u = N, h_x, h_u, s_x, s_u
        self.term = term
        self.z0_p = cp.Parameter(n);  self.zr_p = cp.Parameter(n)
        self.vr_p = cp.Parameter(m)
        self.hz_p = [cp.Parameter(Hx.shape[0]) for _ in range(N)]
        self.hv_p = [cp.Parameter(Hu.shape[0]) for _ in range(N)]
        self.gf_p = cp.Parameter(term.G.shape[0])
        z = cp.Variable((N + 1, n));  v = cp.Variable((N, m))
        self._z, self._v = z, v
        cost = 0.;  cons = [z[0] == self.z0_p]
        for j in range(N):
            cost += (cp.quad_form(z[j] - self.zr_p, Q_c)
                     + cp.quad_form(v[j] - self.vr_p, R_c))
            cons += [z[j+1] == A @ z[j] + B @ v[j],
                     Hx @ z[j] <= self.hz_p[j],
                     Hu @ v[j] <= self.hv_p[j]]
        cost += cp.quad_form(z[N] - self.zr_p, P_f)
        cons.append(term.G @ (z[N] - self.zr_p) <= self.gf_p)
        self.prob = cp.Problem(cp.Minimize(cost), cons)
        print(f"  OK  {name} OCP: N={N}, QP")

    def solve(self, z_k, z_ref, v_ref, aseq):
        self.z0_p.value = z_k;  self.zr_p.value = z_ref;  self.vr_p.value = v_ref
        for j in range(self.N):
            self.hz_p[j].value = self.h_x - aseq[j]*self.s_x
            self.hv_p[j].value = self.h_u - aseq[j]*self.s_u
        self.gf_p.value = self.term.rhs(z_ref, v_ref)
        # The OCP is a convex QP.  Clarabel (interior point) is used for
        # accuracy; OSQP is a fallback.
        try:
            self.prob.solve(solver=cp.CLARABEL, verbose=False)
        except cp.error.SolverError:
            pass
        if self.prob.status not in ("optimal", "optimal_inaccurate"):
            self.prob.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8,
                            max_iter=200_000, polish=True, verbose=False)
        ok = self.prob.status in ("optimal", "optimal_inaccurate")
        return (self._z.value.copy(), self._v.value.copy(), True) if ok \
               else (None, None, False)


# ══════════════════════════════════════════════════════════════════════════════
#  SIMULATION (Algorithm 1)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class SimResult:
    x:       np.ndarray
    z:       np.ndarray
    u:       np.ndarray
    v:       np.ndarray     # nominal input v_{0|k}
    alpha:   np.ndarray
    y_ref:   np.ndarray
    viol_x:  np.ndarray
    viol_u:  np.ndarray
    n_assum: int        # steps with alpha_k > 1 (assumption violation)
    cost:    float
    ok:      bool


def sample_w(rng, Sigma_w, Gamma_w_inv):
    """Gaussian sample conditioned on w in Ell(Gamma_w) (Assumption 1)."""
    while True:
        w = rng.multivariate_normal(np.zeros(Sigma_w.shape[0]), Sigma_w)
        if w @ Gamma_w_inv @ w <= 1.:
            return w


def run_mpc(A, B, A_hat, B_hat, K, C_out, Hx, h_x, Hu, h_u,
            unc, rpi, ocp, y_ref_sig, x0, N, Q_c, R_c,
            homothetic, rng_seed=None):
    rng = np.random.RandomState(rng_seed)
    Gw_inv = la.inv(unc.Gamma_w)
    T = len(y_ref_sig);  n = A.shape[0];  m = B.shape[1]
    Xt = np.zeros((T, n)); Zt = np.zeros((T, n)); Ut = np.zeros((T, m))
    Vt = np.zeros((T, m))
    At = np.zeros(T); vx = np.zeros(T, bool); vu = np.zeros(T, bool)
    ok = True;  n_assum = 0;  cost = 0.
    x_k = x0.copy();  z_k = x0.copy()          # z_0 = x_0  (Assumption 3)
    for k in range(T):
        zr_k, vr_k = steady_state(A, B, C_out, float(y_ref_sig[k]))
        e_k = x_k - z_k
        alpha_k = float(np.sqrt(max(float(e_k @ rpi.P_star_inv @ e_k), 0.)))
        if alpha_k > 1. + 1e-6:                  # should never happen
            n_assum += 1
        At[k] = alpha_k
        aseq = (alpha_seq(min(alpha_k, 1.), rpi.c_S, N) if homothetic
                else np.ones(N + 1))
        z_opt, v_opt, feas = ocp.solve(z_k, zr_k, vr_k, aseq)
        if not feas:
            ok = False;  break
        u_k = v_opt[0] + K @ e_k
        Xt[k] = x_k;  Zt[k] = z_k;  Ut[k] = u_k;  Vt[k] = v_opt[0]
        vx[k] = bool(np.any(Hx @ x_k > h_x + 1e-6))
        vu[k] = bool(np.any(Hu @ u_k > h_u + 1e-6))
        dx = x_k - zr_k;  du = u_k - vr_k
        cost += float(dx @ Q_c @ dx + du @ R_c @ du)
        w_k = sample_w(rng, unc.Sigma_w, Gw_inv)
        x_k = A_hat @ x_k + B_hat @ u_k + w_k
        z_k = A @ z_k + B @ v_opt[0]
    Te = T if ok else k
    return SimResult(x=Xt[:Te], z=Zt[:Te], u=Ut[:Te], v=Vt[:Te],
                     alpha=At[:Te],
                     y_ref=y_ref_sig[:Te], viol_x=vx[:Te], viol_u=vu[:Te],
                     n_assum=n_assum, cost=cost, ok=ok)


def time_to_lane(res, y_target, T_lc, Ts, tol):
    idx = np.where(np.abs(res.x[T_lc:, 0] - y_target) <= tol)[0]
    return (T_lc + idx[0])*Ts if len(idx) else np.nan


# ══════════════════════════════════════════════════════════════════════════════
#  FIGURE
# ══════════════════════════════════════════════════════════════════════════════

def fig_lane_change(res_p, res_r, rpi, vx0, Ts, ey_max, lane_width=3.5,
                    fname="fig_lane_change_comparison_alt.png"):
    """
    Lateral position along the road with the projected cross-sections by
    which each scheme tightens the constraints of the current step:
      HT-MPC: alpha_k * Omega   (alpha_k = ||e_k||_{P^{-1}})
      RT-MPC: Omega             (alpha = 1)
    """
    T_p = len(res_p.x);  T_r = len(res_r.x)
    x_lon_p = np.arange(T_p)*Ts*vx0
    x_lon_r = np.arange(T_r)*Ts*vx0
    s_e = float(np.sqrt(rpi.P_star[0, 0]))
    alp_p = np.clip(res_p.alpha, 0., 1.)

    fig, ax = plt.subplots(1, 1, figsize=(ONE_COL, ONE_COL*0.6), dpi=300)

    # ── Road and constraints ─────────────────────────────────────────────────
    ax.axhline( ey_max, color=C["constr"], lw=1.0, zorder=1)
    ax.axhline(-ey_max, color=C["constr"], lw=1.0, zorder=1)
    ax.axhline(0., color=C["constr"], lw=0.5, alpha=0.35, ls=":", zorder=1)

    # ── Projected cross-sections ─────────────────────────────────────────────
    ax.fill_between(x_lon_r, res_r.z[:, 0] - s_e, res_r.z[:, 0] + s_e,
                    color=C["r"], alpha=0.15, lw=0., zorder=2)
    ax.fill_between(x_lon_p, res_p.z[:, 0] - alp_p*s_e,
                    res_p.z[:, 0] + alp_p*s_e,
                    color=C["p"], alpha=0.30, lw=0., zorder=3)

    # ── Reference and trajectories ───────────────────────────────────────────
    ax.step(x_lon_p, res_p.y_ref, color=C["ref"], lw=0.4, where="post",
            zorder=6)
    ax.plot(x_lon_r, res_r.x[:, 0], color=C["r"], lw=0.6, ls="--", zorder=4)
    ax.plot(x_lon_p, res_p.x[:, 0], color=C["p"], lw=0.6, ls="--", zorder=5)

    # ── Constraint violation markers (none expected) ────────────────────────
    for key, res, x_lon in (("r", res_r, x_lon_r), ("p", res_p, x_lon_p)):
        vi = np.where(res.viol_x)[0]
        if len(vi) > 0:
            ax.scatter(x_lon[vi], res.x[vi, 0], c=C[key], s=18, marker="x",
                       zorder=7)

    # ── Axes ─────────────────────────────────────────────────────────────────
    ax.set_ylabel(r"$e_y$ [m]", labelpad=1)
    ax.set_xlabel(r"Longitudinal distance [m]", labelpad=1)
    ax.set_ylim(-ey_max - 0.35, ey_max + 0.35)
    ax.set_xlim([0., max(x_lon_p[-1], x_lon_r[-1])])
    ax.set_yticks([-lane_width/2, 0., lane_width/2])
    ax.set_yticklabels([f"{-lane_width/2:.2f}", "0", f"{lane_width/2:.2f}"])
    ax.text(2., -lane_width/2 - 0.25, "right lane", color=C["constr"],
            fontsize=FP - 2, va="top")
    ax.text(2.,  lane_width/2 + 0.25, "left lane", color=C["constr"],
            fontsize=FP - 2, va="bottom")

    # ── Inset zoom (steady-state cruise) ─────────────────────────────────────
    axins = inset_axes(ax, width=1., height=0.325,
                       bbox_to_anchor=(42.5, -0.1),
                       bbox_transform=ax.transData,
                       loc="upper left", borderpad=0.)
    for sp in axins.spines.values():
        sp.set_linewidth(0.4); sp.set_color("0.4"); sp.set_linestyle("-.")
    axins.fill_between(x_lon_r, res_r.z[:, 0] - s_e, res_r.z[:, 0] + s_e,
                       color=C["r"], alpha=0.15, lw=0.)
    axins.fill_between(x_lon_p, res_p.z[:, 0] - alp_p*s_e,
                       res_p.z[:, 0] + alp_p*s_e,
                       color=C["p"], alpha=0.30, lw=0.)
    axins.step(x_lon_p, res_p.y_ref, color=C["ref"], lw=0.4, where="post")
    axins.plot(x_lon_r, res_r.x[:, 0], color=C["r"], lw=0.6, ls="--")
    axins.plot(x_lon_p, res_p.x[:, 0], color=C["p"], lw=0.6, ls="--")
    axins.set_xlim(52.5, 72.5);  axins.set_ylim(1.25, 2.5)
    axins.set_xticks([]);  axins.set_yticks([])
    pp, p1, p2 = mark_inset(ax, axins, loc1=4, loc2=2,
                            ec="0.4", linewidth=0.4, linestyle="-.")
    pp.set_alpha(0.4); pp.set_linewidth(0.4)
    for c in (p1, p2):
        c.set_linewidth(0.4); c.set_linestyle("-."); c.set_alpha(0.4)

    # ── Legend ───────────────────────────────────────────────────────────────
    legend_handles = [
        Line2D([0], [0], color=C["ref"], lw=0.8, ls="-", label="Reference"),
        Line2D([0], [0], color=C["p"],   lw=0.9, ls="--", label=LABELS["p"]),
        Line2D([0], [0], color=C["r"],   lw=0.9, ls="--", label=LABELS["r"]),
        Patch(facecolor=C["p"], alpha=0.30, label=r"$\alpha_k\Omega$ (HT-MPC)"),
        Patch(facecolor=C["r"], alpha=0.15, label=r"$\Omega$ (RT-MPC)"),
        Line2D([0], [0], color=C["constr"], lw=1.0, label="Constraint"),
    ]
    ax.legend(handles=legend_handles, loc="upper center",
              bbox_to_anchor=(0.5, 1.325), ncol=3, fontsize=FP - 1,
              frameon=False)

    fig.tight_layout()
    fig.savefig(fname, dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved  {fname}")


# ══════════════════════════════════════════════════════════════════════════════
#  BASELINE: HOMOTHETIC TUBE MPC WITH THE SCALINGS AS DECISION VARIABLES
#
#  S. V. Rakovic, B. Kouvaritakis, R. Findeisen and M. Cannon, "Homothetic tube
#  model predictive control", Automatica, vol. 48, no. 8, pp. 1631-1638, 2012.
#
#  Everything is identical to TubeOCP above - same model, same Omega, same
#  support values s_x, s_u, same cost, same terminal cost and terminal set, same
#  horizon, same measured alpha_{0|k} - except that the tube scalings
#  alpha_{j|k} are DECISION VARIABLES instead of the closed-form sequence of
#  Prop. 3.  They are tied together by the one-step inclusion
#
#      A_K (alpha_j Omega) (+) S_bar  subset of  alpha_{j+1} Omega
#   <=>  alpha_{j+1} >= beta(alpha_j) = sqrt((1 - c_S) alpha_j^2 + c_S),
#
#  i.e. by the very recursion whose explicit solution is Prop. 3.  Written as
#  ||(sqrt(1-c_S) alpha_j, sqrt(c_S))||_2 <= alpha_{j+1} this is a second-order
#  cone constraint, so the baseline solves an SOCP where the proposed scheme
#  solves a QP of the same size minus the N+1 scaling variables.
# ══════════════════════════════════════════════════════════════════════════════

import time

class HomotheticTubeOCPVar:
    """Baseline of Rakovic et al. (2012): alpha_{j|k} are decision variables."""

    def __init__(self, A, B, Hx, h_x, Hu, h_u, s_x, s_u, term, N,
                 Q_c, R_c, P_f, c_S, name):
        n = A.shape[0];  m = B.shape[1]
        self.N, self.term = N, term
        self.z0_p = cp.Parameter(n);  self.zr_p = cp.Parameter(n)
        self.vr_p = cp.Parameter(m)
        self.a0_p = cp.Parameter(nonneg=True)           # measured alpha_{0|k}
        self.gf_p = cp.Parameter(term.G.shape[0])
        z = cp.Variable((N + 1, n));  v = cp.Variable((N, m))
        al = cp.Variable(N + 1, nonneg=True)            # the tube scalings
        self._z, self._v, self._al = z, v, al
        b1, b0 = float(np.sqrt(1. - c_S)), float(np.sqrt(c_S))
        cost = 0.
        cons = [z[0] == self.z0_p, al[0] == self.a0_p, al <= 1.]
        for j in range(N):
            cost += (cp.quad_form(z[j] - self.zr_p, Q_c)
                     + cp.quad_form(v[j] - self.vr_p, R_c))
            cons += [z[j+1] == A @ z[j] + B @ v[j],
                     Hx @ z[j] + al[j]*s_x <= h_x,      # tightening, same s_x
                     Hu @ v[j] + al[j]*s_u <= h_u,      # tightening, same s_u
                     cp.SOC(al[j+1],                    # alpha_{j+1} >= beta(alpha_j)
                            cp.hstack([b1*al[j], cp.Constant(b0)]))]
        cost += cp.quad_form(z[N] - self.zr_p, P_f)
        cons.append(term.G @ (z[N] - self.zr_p) <= self.gf_p)
        self.prob = cp.Problem(cp.Minimize(cost), cons)
        print(f"  OK  {name} OCP: N={N}, SOCP "
              f"({self.prob.size_metrics.num_scalar_variables} scalar variables)")

    def solve(self, z_k, z_ref, v_ref, aseq):
        """Same signature as TubeOCP.solve; only aseq[0] (the measured scale) is
        used, the rest of the sequence is left to the optimiser."""
        self.z0_p.value = z_k;  self.zr_p.value = z_ref;  self.vr_p.value = v_ref
        self.a0_p.value = float(np.clip(aseq[0], 0., 1.))
        self.gf_p.value = self.term.rhs(z_ref, v_ref)
        try:
            self.prob.solve(solver=cp.CLARABEL, verbose=False)
        except cp.error.SolverError:
            pass
        if self.prob.status not in ("optimal", "optimal_inaccurate"):
            self.prob.solve(solver=cp.SCS, eps=1e-9, max_iters=200_000, verbose=False)
        ok = self.prob.status in ("optimal", "optimal_inaccurate")
        return (self._z.value.copy(), self._v.value.copy(), True) if ok \
               else (None, None, False)


class TimedOCP:
    """Wraps an OCP and times every solve: the wall-clock time of the call (what
    the controller actually spends, canonicalisation included) and the time the
    solver itself reports."""

    def __init__(self, ocp, label, kind):
        self.ocp, self.label, self.kind = ocp, label, kind
        self.t_call = [];  self.t_solver = []

    @property
    def n_var(self):
        return self.ocp.prob.size_metrics.num_scalar_variables

    def warm_up(self, *args):
        """One throw-away solve so that canonicalisation is not timed."""
        self.ocp.solve(*args);  self.t_call.clear();  self.t_solver.clear()

    def solve(self, *args):
        t0 = time.perf_counter()
        out = self.ocp.solve(*args)
        self.t_call.append((time.perf_counter() - t0)*1e3)
        st = getattr(self.ocp.prob, "solver_stats", None)
        self.t_solver.append(st.solve_time*1e3 if (st is not None and st.solve_time)
                             else np.nan)
        return out

    def stats(self):
        c = np.array(self.t_call);  s = np.array(self.t_solver, dtype=float)
        return dict(median=np.median(c), mean=c.mean(), max=c.max(), total=c.sum(),
                    solver_median=np.nanmedian(s), solver_max=np.nanmax(s),
                    solver_total=np.nansum(s))


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    SEP = "="*68
    print(SEP + "\n  Lateral Vehicle Lane Change  —  HT-MPC vs RT-MPC\n" + SEP)

    # ── Vehicle model ─────────────────────────────────────────────────────────
    vx0   = 60.0/3.6     # 60 km/h
    Ts    = 0.1
    T_sim = 50
    T_lc  = 12
    A, B  = discretize(vx0, Ts)
    n, m  = 4, 1
    C_out = np.array([[1., 0., 0., 0.]])

    # ── Ancillary gain (Assumption 2) ─────────────────────────────────────────
    Q_lqr = np.diag([50., 10., 1., 10.]);  R_lqr = np.diag([1.0])
    P_lqr = la.solve_discrete_are(A, B, Q_lqr, R_lqr)
    K     = -la.solve(R_lqr + B.T @ P_lqr @ B, B.T @ P_lqr @ A)
    A_K   = A + B @ K
    rho   = float(np.max(np.abs(la.eigvals(A_K))))
    det   = float(abs(la.det(A_K)))
    print(f"  rho(A_K)={rho:.4f}  |det(A_K)|={det:.2e}  "
          f"min|eig(A_K)|={np.min(np.abs(la.eigvals(A_K))):.4f}")
    assert rho < 1.0 and det > 1e-9

    # ── Uncertainty (Assumption 1) ────────────────────────────────────────────
    sigma_vx = 0.1
    p_conf   = 0.99
    D_A, D_B = d_dvx(vx0, Ts)
    eps_reg  = (sigma_vx*0.01)**2
    Sigma_DA = sigma_vx**2*np.outer(D_A.flatten(), D_A.flatten()) + eps_reg*np.eye(n*n)
    Sigma_DB = sigma_vx**2*np.outer(D_B.flatten(), D_B.flatten()) + eps_reg*np.eye(n*m)
    Sigma_w  = np.diag([1e-4, 2e-5, 1e-4, 5e-5])*2
    dv_max   = np.sqrt(stats.chi2.ppf(p_conf, df=1))*sigma_vx
    print(f"  speed bound |Delta_vx| <= {dv_max:.3f} m/s")

    # ── Constraints ───────────────────────────────────────────────────────────
    EY_MAX    = 3.0                    # |e_y| <= EY_MAX (lane centres at +-1.75 m)
    x_bounds  = np.array([EY_MAX, 0.30, 2.0, 1.5])
    u_bounds  = np.array([0.20])
    Hx = np.vstack([np.eye(n), -np.eye(n)]);  h_x = np.tile(x_bounds, 2)
    Hu = np.vstack([np.eye(m), -np.eye(m)]);  h_u = np.tile(u_bounds, 2)

    # ── Stage cost and terminal cost (paper: Lyapunov equation of A_K) ────────
    Q_c = np.diag([20., 5., 0.5, 0.5]);  R_c = np.diag([0.1])
    P_f = la.solve_discrete_lyapunov(A_K.T, Q_c + K.T @ R_c @ K)
    P_f = 0.5*(P_f + P_f.T)
    dec = A_K.T @ P_f @ A_K - P_f + Q_c + K.T @ R_c @ K
    print(f"  terminal decrease residual: {np.max(np.abs(dec)):.1e}")

    # ── Offline design (shared by both schemes) ──────────────────────────────
    print("\n[Offline]")
    unc   = UncertaintyModel(Sigma_DA, Sigma_DB, Sigma_w,
                             p_conf, p_conf, p_conf, n, m)
    QA, QB = compute_Qmax(unc, Hx, h_x, Hu, h_u)
    rpi   = compute_rpi(A_K, unc, QA, QB)
    s_x, s_u = support_values(K, rpi.P_star, Hx, Hu)
    assert np.all(s_x < h_x) and np.all(s_u < h_u), "Assumption 3 violated"
    print(f"  s_x={np.round(s_x[:n], 4)}  s_u={s_u[0]:.4f}")
    print(f"  RT-MPC steering bound: {u_bounds[0]-s_u[0]:.4f} rad "
          f"(of {u_bounds[0]})")
    print(f"  example alpha_seq(0.15) = {np.round(alpha_seq(0.15, rpi.c_S, 4), 4)}")

    lane = 3.5
    y_right, y_left = -lane/2., lane/2.
    setpoints = [steady_state(A, B, C_out, y) for y in (y_right, y_left)]
    term = TerminalSet(A_K, K, Hx, h_x, Hu, h_u, s_x, s_u, setpoints)

    # ── OCPs ─────────────────────────────────────────────────────────────────
    N = 25
    ocp_p = TubeOCP(A, B, Hx, h_x, Hu, h_u, s_x, s_u, term, N,
                    Q_c, R_c, P_f, "HT-MPC")
    ocp_r = TubeOCP(A, B, Hx, h_x, Hu, h_u, s_x, s_u, term, N,
                    Q_c, R_c, P_f, "RT-MPC")
    ocp_b = HomotheticTubeOCPVar(A, B, Hx, h_x, Hu, h_u, s_x, s_u, term, N,
                                 Q_c, R_c, P_f, rpi.c_S, "HT-MPC (opt. scalings)")

    # ── True plant (one fixed draw inside the bound) ─────────────────────────
    rng_plant = np.random.RandomState(42)
    Delta_vx  = rng_plant.normal(0., sigma_vx)
    assert abs(Delta_vx) <= dv_max
    A_hat, B_hat = discretize(vx0 + Delta_vx, Ts)
    print(f"\n  Delta_vx={Delta_vx:+.4f} m/s  "
          f"||Delta_A||_F={la.norm(A_hat-A,'fro'):.4f}  "
          f"||Delta_B||={la.norm(B_hat-B):.4f}")

    # ── Lane change ──────────────────────────────────────────────────────────
    y_ref_sig = np.where(np.arange(T_sim) < T_lc, y_right, y_left).astype(float)
    x0 = np.array([y_right, 0., 0., 0.])

    print("\n[Simulate]  three schemes on the same realisation")
    common = (A, B, A_hat, B_hat, K, C_out, Hx, h_x, Hu, h_u,
              unc, rpi)
    zr0, vr0 = steady_state(A, B, C_out, y_right)
    sch = [TimedOCP(ocp_p, "HT-MPC (closed-form scaling)", "QP"),
           TimedOCP(ocp_b, "HT-MPC (optimised scalings)", "SOCP"),
           TimedOCP(ocp_r, "RT-MPC (rigid tube)", "QP")]
    for s_ in sch:            # one throw-away solve: canonicalisation is not timed
        s_.warm_up(x0, zr0, vr0, np.ones(N + 1))
    # the closed loop is deterministic, so it is replayed a few times and the
    # timing statistics are taken over all the solves (a single pass of 50 solves
    # is a noisy microbenchmark)
    TIMING_REPEATS = 3
    for _rep in range(TIMING_REPEATS):
        res_p = run_mpc(*common, sch[0], y_ref_sig, x0, N, Q_c, R_c,
                        homothetic=True,  rng_seed=7)
        res_b = run_mpc(*common, sch[1], y_ref_sig, x0, N, Q_c, R_c,
                        homothetic=True,  rng_seed=7)
        res_r = run_mpc(*common, sch[2], y_ref_sig, x0, N, Q_c, R_c,
                        homothetic=False, rng_seed=7)

    tol = 0.1*lane
    for lbl, res in (("HT-MPC (closed form)", res_p),
                     ("HT-MPC (opt. scalings)", res_b),
                     ("RT-MPC", res_r)):
        print(f"\n  {lbl}:  status={'OK' if res.ok else 'FAILED'}  "
              f"viol_x={int(res.viol_x.sum())}  viol_u={int(res.viol_u.sum())}  "
              f"alpha>1 steps={res.n_assum}")
        print(f"    closed-loop cost = {res.cost:.1f}")
        print(f"    max |delta|      = {np.abs(res.u).max():.4f} rad")
        print(f"    reaches new lane (+-{tol:.2f} m) at "
              f"t = {time_to_lane(res, y_left, T_lc, Ts, tol):.1f} s")
        print(f"    mean alpha_k     = {res.alpha.mean():.4f}")

    # ── cost and computation ─────────────────────────────────────────────────
    print(f"\n[Comparison]  cost, and computation per OCP solve over "
          f"{len(sch[0].t_call)} solves")
    print(f"  {'scheme':32s}{'class':>6s}{'vars':>6s}{'cost':>10s}{'vs RT':>9s}"
          f"{'solver [ms]':>19s}{'call [ms]':>19s}")
    print(f"  {'':32s}{'':>6s}{'':>6s}{'':>10s}{'':>9s}"
          f"{'median':>10s}{'max':>9s}{'median':>10s}{'max':>9s}")
    for s_, res in zip(sch, (res_p, res_b, res_r)):
        st = s_.stats()
        rel = "-" if res is res_r else f"{100*(1 - res.cost/res_r.cost):+.1f} %"
        print(f"  {s_.label:32s}{s_.kind:>6s}{s_.n_var:>6d}{res.cost:>10.1f}"
              f"{rel:>9s}{st['solver_median']:>10.2f}{st['solver_max']:>9.2f}"
              f"{st['median']:>10.1f}{st['max']:>9.1f}")
    sp_, sb_ = sch[0].stats(), sch[1].stats()
    print("\n  closed-form scaling vs optimised scalings (same realisation):")
    print(f"    relative cost difference : "
          f"{abs(res_p.cost - res_b.cost)/res_b.cost:.2e}")
    print(f"    max |e_y| difference     : "
          f"{np.abs(res_p.x[:, 0] - res_b.x[:, 0]).max():.2e} m")
    print(f"    max |delta| difference   : "
          f"{np.abs(res_p.u - res_b.u).max():.2e} rad")
    _red = 100*(1. - sp_['solver_median']/sb_['solver_median'])
    print(f"    solver time per step     : {sb_['solver_median']:.2f} ms -> "
          f"{sp_['solver_median']:.2f} ms   "
          f"({_red:.0f} % less, {sb_['solver_median']/sp_['solver_median']:.2f} x faster)")
    print(f"    solver time per run      : "
          f"{sb_['solver_total']/TIMING_REPEATS:.0f} ms -> "
          f"{sp_['solver_total']/TIMING_REPEATS:.0f} ms   "
          f"({T_sim} steps of the closed loop)")
    print(f"    call time per step       : {sb_['median']:.1f} ms -> "
          f"{sp_['median']:.1f} ms")
    print(f"    NOTE  the call time is what cvxpy spends per solve; at this problem")
    print(f"          size it is dominated by canonicalisation "
          f"(about {sp_['median']-sp_['solver_median']:.0f} ms, paid by both schemes),")
    print(f"          which hides the difference between the QP and the SOCP.  The")
    print(f"          solver time is what an embedded implementation would pay.")

    print("\n[Figure]")
    fig_lane_change(res_p, res_r, rpi, vx0, Ts, EY_MAX, lane_width=lane)
    print("\n" + SEP + "\n  Done.\n" + SEP)


  Lateral Vehicle Lane Change  —  HT-MPC vs RT-MPC
  rho(A_K)=0.6853  |det(A_K)|=3.19e-04  min|eig(A_K)|=0.0026
  speed bound |Delta_vx| <= 0.258 m/s
  terminal decrease residual: 2.5e-14

[Offline]
  OK  chi2_DA=6.635  chi2_DB=6.635  chi2_w=13.277
  OK  tr(Q^A_max)=3.346e-04  tr(Q^B_max)=8.685e-06
  OK  tau*=0.7057  c_S=0.999999  sqrt(P*[0,0])=0.3549 m
  s_x=[0.3549 0.1254 0.6245 0.8175]  s_u=0.1610
  RT-MPC steering bound: 0.0390 rad (of 0.2)
  example alpha_seq(0.15) = [0.15 1.   1.   1.   1.  ]
  OK  terminal set: t*=3, 40 rows
  OK  HT-MPC OCP: N=25, QP
  OK  RT-MPC OCP: N=25, QP
  OK  HT-MPC (opt. scalings) OCP: N=25, SOCP (155 scalar variables)

  Delta_vx=+0.0497 m/s  ||Delta_A||_F=0.0060  ||Delta_B||=0.0073

[Simulate]  three schemes on the same realisation

  HT-MPC (closed form):  status=OK  viol_x=0  viol_u=0  alpha>1 steps=0
    closed-loop cost = 1481.0
    max |delta|      = 0.1655 rad
    reaches new lane (+-0.35 m) at t = 2.6 s
    mean alpha_k     = 0.1368

  HT-MPC (